In [1]:
!git clone https://github.com/mociatto/AT-SPGD.git

Cloning into 'AT-SPGD'...
remote: Enumerating objects: 343, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 343 (delta 5), reused 17 (delta 5), pack-reused 318 (from 1)
Receiving objects: 100% (343/343), 74.68 KiB | 7.47 MiB/s, done.
Resolving deltas: 100% (172/172), done.


In [2]:
%cd AT-SPGD

/kaggle/working/AT-SPGD


In [3]:
from __future__ import annotations

In [4]:
from pathlib import Path
import sys


def find_project_root() -> Path:
    starts = [Path.cwd()]
    if "__file__" in globals():
        starts.append(Path(__file__).resolve().parent)

    for start in starts:
        for candidate in [start, *start.parents]:
            if (candidate / "src").is_dir():
                return candidate

    for candidate in Path.cwd().iterdir():
        if candidate.is_dir() and (candidate / "src").is_dir():
            return candidate

    raise RuntimeError("Unable to locate project root containing src/.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [5]:
from typing import Any, Dict, List

import pandas as pd
import torch
from IPython.display import display

from src.engine.trainer import default_device, run_standard_training, set_seed

In [6]:
DATASETS = ["cifar10", "cifar100", "svhn", "gtsrb"]
MODELS = ["swin_tiny_patch4_window7_224", "resnet18", "mobilenet_v2", "vit_base_patch16_224"]

EPOCHS = 10
BATCH_SIZE = 128
LR = 1e-3
NUM_WORKERS = 4
SEED = 42

WORK_DIR = PROJECT_ROOT
DATA_ROOT = WORK_DIR / "data"
RESULTS_DIR = WORK_DIR / "results" / "csv"
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
BASELINE_CSV = RESULTS_DIR / "01_baseline_metrics.csv"

In [7]:
def run_experiments() -> pd.DataFrame:
    set_seed(SEED)
    device = default_device()
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    rows: List[Dict[str, Any]] = []

    for dataset_name in DATASETS:
        for model_name in MODELS:
            row = run_standard_training(
                dataset_name=dataset_name,
                model_name=model_name,
                data_root=DATA_ROOT,
                checkpoint_dir=CHECKPOINT_DIR,
                batch_size=BATCH_SIZE,
                epochs=EPOCHS,
                lr=LR,
                num_workers=NUM_WORKERS,
                seed=SEED,
                device=device,
            )
            rows.append(row)
            transfer_file = Path(row["transfer_file"])
            if transfer_file.exists():
                print(f"Transfer batch saved: {transfer_file}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame(rows)


results_df = run_experiments()

100%|██████████| 170M/170M [00:23<00:00, 7.37MB/s]


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/linear.py:134: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:270.)
  return F.linear(input, self.weight, self.bias)


Transfer batch saved: /kaggle/working/AT-SPGD/checkpoints/01_transfer_cifar10_swin_tiny_patch4_window7_224.pt
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 212MB/s]


...

Transfer batch saved: /kaggle/working/AT-SPGD/checkpoints/01_transfer_cifar100_swin_tiny_patch4_window7_224.pt
Transfer batch saved: /kaggle/working/AT-SPGD/checkpoints/01_transfer_cifar100_resnet18.pt
Transfer batch saved: /kaggle/working/AT-SPGD/checkpoints/01_transfer_cifar100_mobilenet_v2.pt
Transfer batch saved: /kaggle/working/AT-SPGD/checkpoints/01_transfer_cifar100_vit_base_patch16_224.pt


100%|██████████| 182M/182M [00:13<00:00, 13.0MB/s]
100%|██████████| 64.3M/64.3M [00:06<00:00, 10.3MB/s]


Transfer batch saved: /kaggle/working/AT-SPGD/checkpoints/01_transfer_svhn_swin_tiny_patch4_window7_224.pt
Transfer batch saved: /kaggle/working/AT-SPGD/checkpoints/01_transfer_svhn_resnet18.pt
Transfer batch saved: /kaggle/working/AT-SPGD/checkpoints/01_transfer_svhn_mobilenet_v2.pt
Transfer batch saved: /kaggle/working/AT-SPGD/checkpoints/01_transfer_svhn_vit_base_patch16_224.pt


100%|██████████| 187M/187M [00:09<00:00, 19.0MB/s]
100%|██████████| 89.0M/89.0M [00:08<00:00, 11.0MB/s]
100%|██████████| 99.6k/99.6k [00:00<00:00, 340kB/s]


Transfer batch saved: /kaggle/working/AT-SPGD/checkpoints/01_transfer_gtsrb_swin_tiny_patch4_window7_224.pt
Transfer batch saved: /kaggle/working/AT-SPGD/checkpoints/01_transfer_gtsrb_resnet18.pt
Transfer batch saved: /kaggle/working/AT-SPGD/checkpoints/01_transfer_gtsrb_mobilenet_v2.pt
Transfer batch saved: /kaggle/working/AT-SPGD/checkpoints/01_transfer_gtsrb_vit_base_patch16_224.pt


In [8]:
results_df.to_csv(BASELINE_CSV, index=False)
display(results_df)

,dataset,model,num_classes,best_checkpoint,transfer_file,accuracy,macro_f1,macro_auroc
0,cifar10,swin_tiny_patch4_window7_224,10,/kaggle/working/AT-SPGD/checkpoints/01_baselin...,/kaggle/working/AT-SPGD/checkpoints/01_transfe...,0.935100,0.935025,0.997218
1,cifar10,resnet18,10,/kaggle/working/AT-SPGD/checkpoints/01_baselin...,/kaggle/working/AT-SPGD/checkpoints/01_transfe...,0.889700,0.890093,0.993169
2,cifar10,mobilenet_v2,10,/kaggle/working/AT-SPGD/checkpoints/01_baselin...,/kaggle/working/AT-SPGD/checkpoints/01_transfe...,0.886300,0.885957,0.993114
3,cifar10,vit_base_patch16_224,10,/kaggle/working/AT-SPGD/checkpoints/01_baselin...,/kaggle/working/AT-SPGD/checkpoints/01_transfe...,0.944900,0.945011,0.997889
4,cifar100,swin_tiny_patch4_window7_224,100,/kaggle/working/AT-SPGD/checkpoints/01_baselin...,/kaggle/working/AT-SPGD/checkpoints/01_transfe...,0.749300,0.751264,0.992893
5,cifar100,resnet18,100,/kaggle/working/AT-SPGD/checkpoints/01_baselin...,/kaggle/working/AT-SPGD/checkpoints/01_transfe...,0.660600,0.658208,0.989856
6,cifar100,mobilenet_v2,100,/kaggle/working/AT-SPGD/checkpoints/01_baselin...,/kaggle/working/AT-SPGD/checkpoints/01_transfe...,0.658200,0.658215,0.988918
7,cifar100,vit_base_patch16_224,100,/kaggle/working/AT-SPGD/checkpoints/01_baselin...,/kaggle/working/AT-SPGD/checkpoints/01_transfe...,0.783700,0.784221,0.994082
8,svhn,swin_tiny_patch4_window7_224,10,/kaggle/working/AT-SPGD/checkpoints/01_baselin...,/kaggle/working/AT-SPGD/checkpoints/01_transfe...,0.662800,0.642552,0.936299
9,svhn,resnet18,10,/kaggle/working/AT-SPGD/checkpoints/01_baselin...,/kaggle/working/AT-SPGD/checkpoints/01_transfe...,0.656999,0.642967,0.934038
